In [2]:
import numpy as np
import heapq
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display
from typing import List, Tuple, Set

# --- PHYSICS AND ENVIRONMENT MODULE ---
class BatteryModel:
    def __init__(self, capacity_ah: float, nominal_voltage: float):
        self.capacity_wh = capacity_ah * nominal_voltage
        self.idle_drain = 0.05      
        self.forward_drain = 0.50   
        self.turn_drain = 1.20      
        
    def get_action_cost(self, action: str, current_soc: float) -> float:
        ocv_penalty = 1.0 + (0.5 * (1.0 - current_soc))
        if action == 'FORWARD': cost = self.forward_drain * ocv_penalty
        elif action in ['TURN_LEFT', 'TURN_RIGHT']: cost = self.turn_drain * ocv_penalty
        elif action == 'WAIT': cost = self.idle_drain * ocv_penalty
        else: cost = 0.0
        return (cost / self.capacity_wh)

class GridEnvironment:
    def __init__(self, width: int, height: int, obstacles: Set[Tuple[int, int]]):
        self.width = width
        self.height = height
        self.obstacles = obstacles
        
    def is_valid(self, x: int, y: int) -> bool:
        if x < 0 or x >= self.width or y < 0 or y >= self.height: return False
        if (x, y) in self.obstacles: return False
        return True

DIRECTIONS = {0: (0, 1), 1: (1, 0), 2: (0, -1), 3: (-1, 0)}

# --- AI SOLVER MODULE: SPACE-TIME A* ---
class RCNode:
    def __init__(self, x, y, theta, time, soc, parent=None, action=None, g_score=0.0):
        self.x, self.y, self.theta, self.time, self.soc = x, y, theta, time, soc
        self.parent, self.action, self.g_score, self.f_score = parent, action, g_score, 0.0

    def __lt__(self, other):
        if self.f_score == other.f_score:
            if self.soc == other.soc: return self.time < other.time
            return self.soc > other.soc
        return self.f_score < other.f_score

def manhattan_distance(x1, y1, x2, y2):
    return abs(x1 - x2) + abs(y1 - y2)

class PrioritizedSpaceTimeAStar:
    """Computes paths while avoiding dynamic reservations from higher-priority agents."""
    def __init__(self, env: GridEnvironment, battery: BatteryModel):
        self.env = env
        self.battery = battery
        
    def search(self, start, goal, init_theta, reservations, edge_reservations, parked_agents, critical_soc_limit=0.05, max_time=150, heuristic_weight=2.0):
        open_list = []
        dominance_dict = {} 
        
        start_node = RCNode(start[0], start[1], init_theta, 0, 1.0, g_score=0.0)
        start_node.f_score = start_node.g_score + (heuristic_weight * manhattan_distance(start[0], start[1], goal[0], goal[1]))
        heapq.heappush(open_list, start_node)
        
        while open_list:
            current = heapq.heappop(open_list)
            if (current.x, current.y) == goal: return self.reconstruct_path(current)
            if current.time >= max_time: continue
                
            state_key = (current.x, current.y, current.theta, current.time)
            if state_key in dominance_dict and dominance_dict[state_key] >= current.soc: continue 
            dominance_dict[state_key] = current.soc

            transitions = [
                ('FORWARD', current.x + DIRECTIONS[current.theta][0], current.y + DIRECTIONS[current.theta][1], current.theta),
                ('TURN_LEFT', current.x, current.y, (current.theta - 1) % 4),
                ('TURN_RIGHT', current.x, current.y, (current.theta + 1) % 4),
                ('WAIT', current.x, current.y, current.theta)
            ]
            
            for action, nx, ny, ntheta in transitions:
                if not self.env.is_valid(nx, ny): continue
                ntime = current.time + 1
                
                # --- SPACE-TIME COLLISION CHECKS ---
                if (nx, ny, ntime) in reservations: continue 
                if (nx, ny, current.x, current.y, ntime) in edge_reservations: continue
                if (nx, ny) in parked_agents and ntime >= parked_agents[(nx, ny)]: continue
                # -----------------------------------
                    
                nsoc = current.soc - self.battery.get_action_cost(action, current.soc)
                if nsoc <= critical_soc_limit: continue 
                
                # Asymmetric Penalties to discourage unnecessary waiting/spinning
                action_cost = 5.0 if action == 'WAIT' else (2.0 if action in ['TURN_LEFT', 'TURN_RIGHT'] else 1.0)
                
                neighbor = RCNode(nx, ny, ntheta, ntime, nsoc, current, action, current.g_score + action_cost)
                neighbor.f_score = neighbor.g_score + (heuristic_weight * manhattan_distance(nx, ny, goal[0], goal[1]))
                heapq.heappush(open_list, neighbor)
                
        return None

    def reconstruct_path(self, node):
        path = []
        while node:
            path.append({'pos': (node.x, node.y), 'theta': node.theta, 'time': node.time, 'soc': node.soc, 'action': node.action})
            node = node.parent
        return path[::-1]

class PrioritizedPlanner:
    def __init__(self, env: GridEnvironment, agents: List[dict]):
        self.env = env
        self.agents = agents
        self.battery = BatteryModel(capacity_ah=5.0, nominal_voltage=24.0)
        self.solver = PrioritizedSpaceTimeAStar(self.env, self.battery)
        
    def solve(self):
        # Sort agents by task difficulty (longest distance gets priority)
        sorted_agents = sorted(self.agents, key=lambda a: manhattan_distance(a['start'][0], a['start'][1], a['goal'][0], a['goal'][1]), reverse=True)
        
        paths = {}
        reservations = set()       # Tracks (x, y, time) 
        edge_reservations = set()  # Tracks (prev_x, prev_y, next_x, next_y, time)
        parked_agents = {}         # Tracks (x, y) -> time_reached
        
        for agent in sorted_agents:
            print(f"Planning route for Agent {agent['id']}...")
            path = self.solver.search(
                agent['start'], agent['goal'], agent['init_theta'], 
                reservations, edge_reservations, parked_agents
            )
            
            if not path:
                print(f"FAILED: Agent {agent['id']} could not find a valid space-time route.")
                return None
                
            paths[agent['id']] = path
            
            for i in range(len(path)):
                loc = path[i]['pos']
                t = path[i]['time']
                reservations.add((loc[0], loc[1], t))
                
                if i > 0:
                    prev_loc = path[i-1]['pos']
                    edge_reservations.add((prev_loc[0], prev_loc[1], loc[0], loc[1], t))
                    
            final_loc = path[-1]['pos']
            final_time = path[-1]['time']
            parked_agents[final_loc] = final_time

        print("All paths successfully mapped and synchronized.")
        return paths

# --- VERIFICATION AND VISUALIZATION MODULE ---
class Visualizer:
    def __init__(self, env, paths):
        self.env = env
        self.paths = paths
        self.max_time = max(len(p) for p in paths.values())
        
        self.fig, self.ax = plt.subplots(figsize=(10, 10))
        self.ax.set_xlim(-0.5, env.width - 0.5)
        self.ax.set_ylim(-0.5, env.height - 0.5)
        self.ax.set_xticks(np.arange(-0.5, env.width, 1))
        self.ax.set_yticks(np.arange(-0.5, env.height, 1))
        self.ax.grid(True, alpha=0.3)
        
        for ox, oy in env.obstacles:
            self.ax.add_patch(plt.Rectangle((ox - 0.5, oy - 0.5), 1, 1, color='#555555'))
            
        self.colors = ['#1f77b4', '#d62728', '#2ca02c', '#9467bd']
        self.agent_patches = {}
        self.soc_texts = {}
        
        for i, agent_id in enumerate(paths.keys()):
            color = self.colors[(agent_id - 1) % len(self.colors)]
            patch = plt.Circle((-1, -1), 0.4, color=color, zorder=3)
            self.ax.add_patch(patch)
            self.agent_patches[agent_id] = patch
            
            txt = self.ax.text(0, 0, '', color=color, fontweight='bold', transform=self.ax.transAxes)
            self.soc_texts[agent_id] = txt

    def init_anim(self):
        for patch in self.agent_patches.values(): patch.center = (-1, -1)
        return list(self.agent_patches.values()) + list(self.soc_texts.values())

    def update(self, frame):
        self.ax.set_title(f"Warehouse Time Step: {frame}", fontsize=14, pad=20)
        y_offset = 1.05
        
        for i, (agent_id, path) in enumerate(self.paths.items()):
            state_idx = min(frame, len(path) - 1)
            state = path[state_idx]
            self.agent_patches[agent_id].center = state['pos']
            
            action = state['action'] if state['action'] else "START"
            self.soc_texts[agent_id].set_text(f"Robo {agent_id}: {state['soc']*100:.1f}% [{action}]")
            self.soc_texts[agent_id].set_position((0.01 + (i * 0.22), y_offset))
            
        return list(self.agent_patches.values()) + list(self.soc_texts.values())

    def animate(self):
        anim = animation.FuncAnimation(
            self.fig, self.update, init_func=self.init_anim,
            frames=self.max_time + 5, interval=400, blit=False, repeat=False
        )
        plt.close(self.fig)
        return HTML(anim.to_jshtml())

if __name__ == "__main__":
    print("Generating 20x20 Warehouse Environment...")
    
    obstacles = set()
    for x in [2, 3, 8, 9, 14, 15]:
        for y in range(2, 18):
            if y not in [9, 10]: 
                obstacles.add((x, y))
                
    env = GridEnvironment(20, 20, obstacles)
    
    agents = [
        {'id': 1, 'start': (0, 5), 'goal': (19, 9), 'init_theta': 1},  
        {'id': 2, 'start': (19, 9), 'goal': (0, 9), 'init_theta': 3}, 
        {'id': 3, 'start': (5, 2), 'goal': (5, 19), 'init_theta': 0},  
        {'id': 4, 'start': (11, 3), 'goal': (11, 0), 'init_theta': 2}  
    ]
    
    print("Initializing Decoupled Prioritized Planner...")
    planner = PrioritizedPlanner(env, agents)
    optimal_paths = planner.solve()
    
    if optimal_paths:
        print("Rendering animation widget...")
        vis = Visualizer(env, optimal_paths)
        display(vis.animate())
    else:
        print("No feasible solution exists.")

Generating 20x20 Warehouse Environment...
Initializing Decoupled Prioritized Planner...
Planning route for Agent 1...
Planning route for Agent 2...
Planning route for Agent 3...
Planning route for Agent 4...
All paths successfully mapped and synchronized.
Rendering animation widget...
